# 1. Two-Stage Pipeline Evaluation: Candidate Budget × Reranker

The reranking notebooks score **logged** slates, the articles a user was actually shown.
In deployment the reranker instead orders what **candidate generation** retrieves. This
notebook evaluates the two stages together and finds the best candidate budget.

- **Candidate generation**: two retrievers. Ranks 1–500 come from the popularity model and
  ranks 501–1250 from the content model, with no article repeated within an impression.
- **Candidate budget**: keep the top `P` popularity and top `C` content candidates
  (ranks `1..P` and `501..500+C`), so `M = P + C` articles go to the reranker.
- **Reranker**: the final `RR_4` model (`+metadata`, softmax + LambdaNDCG, full training set).
  Every candidate gets causal ranking features as of its impression, from a replay of the
  timeline.

Candidates are scored **once** at the largest budget. The reranker scores each candidate
independently, so any smaller budget is just a filter on the same scores.

**Labels are incomplete.** A retrieved article that was never shown has unknown relevance:
it counts as "not a known click", not as a negative. Every known click stays in the
denominator, so clicks the generators miss are misses. `candidate_recall` (the share of
known clicks retrieved at all) caps every ranking metric.

## 1.1 Load Packages

In [1]:
import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from IPython.display import display
from torch import nn

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [2]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
# Processed request/event tables used by the causal feature replay
PROCESSED_ROOT = DATA_ROOT / "processed"

# Candidate generation output: per impression, ranks 1-500 are popularity, 501-1250 content
CANDIDATES = Path("/Users/numkoos/Downloads/recommendations_validation_top1250.parquet")

# Shared evaluator, and the feature-engineering package used to replay causal features
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from rr_evaluation import evaluate_retrieved_candidates
from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle

## 1.3 Load Generated Candidates

One row per (impression, candidate), with `cutoff` giving the candidate's rank within the
generator output.

In [3]:
candidate_summary = (
    pl.scan_parquet(CANDIDATES)
    .select(
        pl.len().alias("rows"),
        pl.col("impression_uid").n_unique().alias("impressions"),
        (pl.col("cutoff") <= 500).sum().alias("popularity_rows"),
        (pl.col("cutoff") > 500).sum().alias("content_rows"),
    )
    .collect()
)
display(candidate_summary)
pl.scan_parquet(CANDIDATES).head(5).collect()

rows,impressions,popularity_rows,content_rows
u32,u32,u32,u32
39530000,31624,15812000,23718000


impression_uid,cutoff,news_code
i64,i16,i32
10154218,1,7821
10154218,2,42977
10154218,3,12029
10154218,4,52723
10154218,5,33885


# 2. Experiment Configuration

Scoring is the expensive step: features are built for every (impression, candidate) pair
during the replay. `EVAL_IMPRESSIONS` scores a fixed random subset of impressions; set it
to `None` for all 31,624 (about 40M pairs at the largest budget). Each budget is evaluated
on the same impressions.

In [4]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using {DEVICE}")

# Reranker to load (an RR_4_branch_interaction output folder)
RERANKER_DIR = ARTIFACTS / "branch_interaction"
RERANKER_NAME = "branch_plus_metadata__softmax_plus_lambda_ndcg__full_data"

# Impressions and the largest budget to score
EVAL_IMPRESSIONS = 10_000        # None = every impression in the candidate file
MAX_POPULARITY_TOP = 500         # popularity ranks 1..500
MAX_CONTENT_TOP = 750            # content ranks 501..1250

# Budgets to evaluate (any combination up to the maxima above)
POPULARITY_BUDGETS = (0, 50, 100, 200, 300, 500)
CONTENT_BUDGETS = (0, 50, 100, 200, 300, 500, 750)
SELECTION_METRIC = "ndcg_at_10"
TOLERANCE = 0.005                # smallest budget within this of the best is also reported

# Must match the settings the reranker was trained with in RR_4
MODEL_DIM = 64
ATTENTION_HEADS = 2
ATTENTION_DROPOUT = 0.10
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20
HISTORY_CAP = 30
SHORT_TERM_CAP = 10
SCORING_BATCH_ROWS = 4096

OUTPUT_DIR = DATA_ROOT / "5_Models" / "5_03_Pipeline_Evaluation" / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
impression_tag = "all" if EVAL_IMPRESSIONS is None else f"{EVAL_IMPRESSIONS}imp"
SCORE_PATH = OUTPUT_DIR / f"{RERANKER_NAME}__pop{MAX_POPULARITY_TOP}_content{MAX_CONTENT_TOP}_{impression_tag}.parquet"
print(f"Score file: {SCORE_PATH}")

Using mps
Score file: /Users/numkoos/Downloads/v3-mindsmall/5_Models/5_03_Pipeline_Evaluation/artifacts/branch_plus_metadata__softmax_plus_lambda_ndcg__full_data__pop500_content750_10000imp.parquet


# 3. Load the Reranker

## 3.1 Article Inputs

The same article tables the reranker was trained with, indexed by `article_row`.

In [5]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["news_code", "article_row", "category_code", "subcategory_code"],
)
news_to_row = dict(zip(article_index["news_code"].to_list(), article_index["article_row"].to_list()))

text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
entity_vectors = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")
title_tokens = np.load(REPRESENTATIONS / "title_token_ids.npy").astype(np.int64)
N_ARTICLES = text_vectors.shape[0]

# Category and subcategory codes per article row (0 = padding / unknown)
article_category = np.zeros(N_ARTICLES, dtype=np.int64)
article_subcategory = np.zeros(N_ARTICLES, dtype=np.int64)
article_category[article_index["article_row"].to_numpy()] = article_index["category_code"].to_numpy()
article_subcategory[article_index["article_row"].to_numpy()] = article_index["subcategory_code"].to_numpy()
print(f"{N_ARTICLES:,} article rows; {article_category.max()} categories; "
      f"{article_subcategory.max()} subcategories; title vocabulary {title_tokens.max() + 1:,}")

51,283 article rows; 17 categories; 257 subcategories; title vocabulary 28,714


## 3.2 Model Components

These cells are copied unchanged from `RR_4_branch_interaction.ipynb`, so the saved
weights load directly. See that notebook for the explanation of each component.

In [6]:
class AdditiveAttention(nn.Module):
    """Pool a sequence of history states into one user vector with learned relevance weights."""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        # states: [batch, history, dim]; padding_mask: [batch, history], True = padding
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False                       # softmax needs one open slot

        scores = torch.tanh(self.proj(states)) @ self.query    # [batch, history]
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)     # [batch, dim]

In [7]:
class NewsEncoder(nn.Module):
    """Map article rows to MODEL_DIM news vectors from the enabled article inputs."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.register_buffer("text", torch.from_numpy(np.array(text_vectors, dtype=np.float32)), persistent=False)
        width = self.text.shape[1]

        if config["metadata"]:
            self.register_buffer("entity", torch.from_numpy(np.array(entity_vectors, dtype=np.float32)), persistent=False)
            self.register_buffer("category", torch.from_numpy(article_category), persistent=False)
            self.register_buffer("subcategory", torch.from_numpy(article_subcategory), persistent=False)
            self.category_embedding = nn.Embedding(int(article_category.max()) + 1, 16, padding_idx=0)
            self.subcategory_embedding = nn.Embedding(int(article_subcategory.max()) + 1, 16, padding_idx=0)
            self.entity_projection = nn.Linear(self.entity.shape[1], 32)
            width += 16 + 16 + 32

        if config["title_encoder"]:
            self.register_buffer("title_tokens", torch.from_numpy(title_tokens), persistent=False)
            self.word_embedding = nn.Embedding(int(title_tokens.max()) + 1, MODEL_DIM, padding_idx=0)
            self.word_attention = nn.MultiheadAttention(MODEL_DIM, ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True)
            self.word_pool = AdditiveAttention(MODEL_DIM)
            width += MODEL_DIM

        self.output = nn.Sequential(nn.Linear(width, MODEL_DIM), nn.LayerNorm(MODEL_DIM))

    def encode_titles(self, rows):
        tokens = self.title_tokens[rows]                       # [articles, title_len]
        padding = tokens.eq(0)
        padding[padding.all(dim=1), 0] = False                 # padding articles keep one slot
        words = self.word_embedding(tokens)
        attended, _ = self.word_attention(words, words, words, key_padding_mask=padding, need_weights=False)
        return self.word_pool(attended, padding)

    def forward(self, rows):
        # Encode each distinct article in the batch once, then scatter back to every position
        unique_rows, inverse = torch.unique(rows, return_inverse=True)
        parts = [self.text[unique_rows]]
        if self.config["metadata"]:
            parts += [
                self.category_embedding(self.category[unique_rows]),
                self.subcategory_embedding(self.subcategory[unique_rows]),
                self.entity_projection(self.entity[unique_rows]),
            ]
        if self.config["title_encoder"]:
            parts.append(self.encode_titles(unique_rows))
        return self.output(torch.cat(parts, dim=-1))[inverse]   # rows.shape + [MODEL_DIM]

In [8]:
class UserEncoder(nn.Module):
    """Encode the click history into per-candidate user vectors."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        if config["position"]:
            self.position_embedding = nn.Embedding(HISTORY_CAP, MODEL_DIM)
        self.self_attention = nn.MultiheadAttention(MODEL_DIM, ATTENTION_HEADS, dropout=ATTENTION_DROPOUT, batch_first=True)
        self.norm1 = nn.LayerNorm(MODEL_DIM)
        self.feed_forward = nn.Sequential(
            nn.Linear(MODEL_DIM, 4 * MODEL_DIM),
            nn.ReLU(),
            nn.Dropout(ATTENTION_DROPOUT),
            nn.Linear(4 * MODEL_DIM, MODEL_DIM),
        )
        self.norm2 = nn.LayerNorm(MODEL_DIM)
        self.pool = AdditiveAttention(MODEL_DIM)
        self.default_user = nn.Parameter(torch.zeros(MODEL_DIM))
        if config["short_term"]:
            self.short_term = nn.GRU(MODEL_DIM, MODEL_DIM, batch_first=True)
            self.gate = nn.Linear(2 * MODEL_DIM, MODEL_DIM)
        self.last_gate = None

    @property
    def output_dim(self):
        return MODEL_DIM * (2 if self.config["candidate_pooling"] else 1)

    def forward(self, history, padding_mask, candidates):
        # history: [B, H, D]; padding_mask: [B, H] (True = padding); candidates: [B, W, D]
        empty = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[empty, -1] = False

        if self.config["position"]:
            # Column j of a left-padded history has recency rank H-1-j (0 = newest)
            ranks = torch.arange(HISTORY_CAP - 1, -1, -1, device=history.device)
            history = history + self.position_embedding(ranks)

        attended, _ = self.self_attention(history, history, history, key_padding_mask=safe_mask, need_weights=False)
        states = self.norm1(history + attended)
        states = self.norm2(states + self.feed_forward(states))
        states = states.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        user = self.pool(states, padding_mask)
        if self.config["short_term"]:
            recent = states[:, -SHORT_TERM_CAP:]                       # newest clicks, zeros for padding
            _, short = self.short_term(recent)
            short = short.squeeze(0)
            gate = torch.sigmoid(self.gate(torch.cat([user, short], dim=1)))
            user = gate * user + (1 - gate) * short
            self.last_gate = gate.detach().mean().item()
        user = torch.where(empty.unsqueeze(1), self.default_user.unsqueeze(0), user)
        user = user.unsqueeze(1).expand(-1, candidates.shape[1], -1)   # [B, W, D]

        if not self.config["candidate_pooling"]:
            return user
        # Candidate-as-query attention over the contextualised history
        scores = candidates @ states.transpose(1, 2) / np.sqrt(MODEL_DIM)       # [B, W, H]
        scores = scores.masked_fill(safe_mask.unsqueeze(1), -1e9)
        weights = torch.softmax(scores, dim=-1).masked_fill(padding_mask.unsqueeze(1), 0.0)
        candidate_user = weights @ states                                       # [B, W, D]
        return torch.cat([user, candidate_user], dim=-1)

In [9]:
class BranchReranker(nn.Module):
    """Score candidates from the enabled tabular, news and user branches."""

    def __init__(self, config, tabular_width):
        super().__init__()
        if config["user_encoder"] and not config["text"]:
            raise ValueError("The user encoder needs the news encoder (text=True)")
        if not (config["tabular"] or config["text"]):
            raise ValueError("At least one of the tabular or text branches must be on")
        self.config = config
        width = tabular_width if config["tabular"] else 0
        if config["text"]:
            self.news_encoder = NewsEncoder(config)
            width += MODEL_DIM
        if config["user_encoder"]:
            self.user_encoder = UserEncoder(config)
            width += self.user_encoder.output_dim

        layers = []
        for hidden in HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, candidate_rows, history_rows):
        # tabular: [B, W, T]; candidate_rows: [B, W]; history_rows: [B, H] → logits [B, W]
        parts = [tabular] if self.config["tabular"] else []
        if self.config["text"]:
            candidates = self.news_encoder(candidate_rows)
            parts.append(candidates)
            if self.config["user_encoder"]:
                history = self.news_encoder(history_rows)
                parts.append(self.user_encoder(history, history_rows.eq(0), candidates))
        return self.scorer(torch.cat(parts, dim=-1)).squeeze(-1)

In [10]:
def encode_histories(impression_ids, profiles):
    """Left-padded [impressions, HISTORY_CAP] article rows with the newest click last."""
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
    return histories

## 3.3 Load Weights and Preprocessing

The tabular preprocessor was fitted in `RR_4` on training data only. The feature schema
check guarantees the replayed features match what the model was trained on.

In [11]:
checkpoint = torch.load(RERANKER_DIR / RERANKER_NAME / "model.pt", map_location=DEVICE, weights_only=False)
model = BranchReranker(checkpoint["config"], checkpoint["tabular_width"]).to(DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
preprocessor = joblib.load(RERANKER_DIR / "preprocessor.joblib")

FEATURES = json.loads((RUN / "feature_schema.json").read_text())["model_columns"]
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != FEATURES:
    raise ValueError("The feature code and the saved feature run disagree on the schema")
print(f"Loaded {RERANKER_NAME}: branches {[name for name, on in checkpoint['config'].items() if on]}, "
      f"loss {checkpoint['loss']}")

Loaded branch_plus_metadata__softmax_plus_lambda_ndcg__full_data: branches ['tabular', 'text', 'metadata', 'user_encoder'], loss softmax+lambda_ndcg


# 4. Score Generated Candidates

The timeline is replayed in timestamp order. Each selected impression's candidates get
features from the state **before** that impression, and candidates never update the
state. Rows are scored in chunks and written to a partial file, which is renamed only
once every selected impression is covered. An existing score file is reused.

In [ ]:
def select_candidates():
    """Candidates for the evaluated impressions, up to the largest budget."""
    budget = (pl.col("cutoff") <= MAX_POPULARITY_TOP) | (
        (pl.col("cutoff") > 500) & (pl.col("cutoff") <= 500 + MAX_CONTENT_TOP)
    )
    candidates = pl.scan_parquet(CANDIDATES).filter(budget)
    if EVAL_IMPRESSIONS is not None:
        ids = candidates.select("impression_uid").unique().sort("impression_uid").collect()["impression_uid"].to_numpy()
        chosen = np.random.default_rng(SEED).choice(ids, size=min(EVAL_IMPRESSIONS, len(ids)), replace=False)
        candidates = candidates.filter(pl.col("impression_uid").is_in(chosen.tolist()))
    return candidates.collect().to_pandas().sort_values(["impression_uid", "cutoff"]).reset_index(drop=True)


def score_rows(frame, histories):
    """Reranker probabilities for feature rows (pointwise, in fixed-size chunks)."""
    uids = frame["impression_uid"].drop_duplicates().tolist()
    history_rows = encode_histories(uids, histories)
    history_index = frame["impression_uid"].map(pd.Series(np.arange(len(uids)), index=uids)).to_numpy()
    tabular = preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
    candidates = frame["news_code"].map(news_to_row).to_numpy(np.int64)

    scores = []
    with torch.no_grad():
        for start in range(0, len(frame), SCORING_BATCH_ROWS):
            stop = start + SCORING_BATCH_ROWS
            logits = model(
                torch.from_numpy(tabular[start:stop]).unsqueeze(1).to(DEVICE),
                torch.from_numpy(candidates[start:stop]).unsqueeze(1).to(DEVICE),
                torch.from_numpy(history_rows[history_index[start:stop]]).to(DEVICE),
            )
            scores.append(torch.sigmoid(logits.squeeze(1)).cpu().numpy())
    return np.concatenate(scores)


def score_generated_candidates():
    """Replay validation, build causal features for the selected candidates and score them."""
    candidates = select_candidates()
    rows_by_impression = candidates.groupby("impression_uid", sort=False).indices
    print(f"Scoring {len(candidates):,} candidates for {len(rows_by_impression):,} impressions")

    profiles = pl.read_parquet(RUN / "request_profiles_validation" / "*.parquet",
                               columns=["impression_uid", "full_history_rows"])
    histories = dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

    bundle = Bundle(DATA_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(PROCESSED_ROOT), DEFAULT_SETTINGS)
    partial_path = SCORE_PATH.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()

    def flush():
        nonlocal writer, pending
        if not pending:
            return
        frame = pd.concat(pending, ignore_index=True)
        pending = []
        frame["score"] = score_rows(frame, histories)
        table = pa.Table.from_pandas(frame[["impression_uid", "news_code", "cutoff", "score"]], preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)

    try:
        for timestamp, requests, events in replay_batches(PROCESSED_ROOT, DEFAULT_SETTINGS, splits=("train", "validation")):
            state.begin(requests, timestamp)
            selected = requests[requests["split"].eq("validation") & requests["impression_uid"].isin(set(rows_by_impression))]
            if len(selected):
                pairs = candidates.iloc[np.concatenate([rows_by_impression[uid] for uid in selected["impression_uid"]])]
                features = build_features(selected, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved")
                pending.append(features.merge(pairs, on=["impression_uid", "news_code"], validate="one_to_one"))
                impressions += len(selected)
                if sum(len(frame) for frame in pending) >= 50_000:
                    flush()
                    print(f"  {impressions:,}/{len(rows_by_impression):,} impressions scored "
                          f"({(time.monotonic() - started) / 60:.1f} min)", flush=True)
            state.finish(requests, events, timestamp, {"train", "validation"})
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(f"Scored {impressions:,} impressions but {len(rows_by_impression):,} were selected")
    partial_path.rename(SCORE_PATH)
    print(f"Done in {(time.monotonic() - started) / 60:.1f} minutes")


if not SCORE_PATH.exists():
    score_generated_candidates()
scores = pl.read_parquet(SCORE_PATH)
print(f"{scores['impression_uid'].n_unique():,} impressions, {scores.height:,} scored candidates")

Scoring 12,500,000 candidates for 10,000 impressions


/var/folders/6w/spf_46751yq2lz6xm8c94jjh0000gn/T/ipykernel_49988/600341168.py:28: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  torch.from_numpy(candidates[start:stop]).unsqueeze(1).to(DEVICE),


  40/10,000 impressions scored (2.4 min)
  80/10,000 impressions scored (2.4 min)
  120/10,000 impressions scored (2.5 min)
  160/10,000 impressions scored (2.5 min)
  200/10,000 impressions scored (2.5 min)
  240/10,000 impressions scored (2.5 min)
  280/10,000 impressions scored (2.5 min)
  320/10,000 impressions scored (2.6 min)
  360/10,000 impressions scored (2.6 min)
  400/10,000 impressions scored (2.6 min)
  440/10,000 impressions scored (2.6 min)
  480/10,000 impressions scored (2.7 min)
  520/10,000 impressions scored (2.7 min)
  560/10,000 impressions scored (2.7 min)
  600/10,000 impressions scored (2.7 min)
  641/10,000 impressions scored (2.7 min)
  681/10,000 impressions scored (2.8 min)
  721/10,000 impressions scored (2.8 min)
  761/10,000 impressions scored (2.8 min)
  801/10,000 impressions scored (2.8 min)
  841/10,000 impressions scored (2.8 min)
  881/10,000 impressions scored (2.9 min)
  921/10,000 impressions scored (2.9 min)
  961/10,000 impressions scored (2.9

# 5. Evaluate Every Candidate Budget

For each budget `(P, C)`, two orderings of the same `M = P + C` candidates are compared:

- **Generator only**: interleave the two retrievers' own rankings (popularity 1, content 1,
  popularity 2, …). This is the pipeline without a reranker.
- **Generator + reranker**: order by the reranker's score.

The ranking metrics are known-click metrics; `candidate_recall` is their ceiling.

In [ ]:
# Rank within each retriever's own block: popularity 1..500, content 1..750
block_rank = pl.when(pl.col("cutoff") <= 500).then(pl.col("cutoff")).otherwise(pl.col("cutoff") - 500)
is_popularity = pl.col("cutoff") <= 500
interleaved = -(2 * block_rank - is_popularity.cast(pl.Int32))   # pop 1, content 1, pop 2, ...

ORDERINGS = {"generator_only": interleaved, "generator_plus_reranker": pl.col("score")}


def in_budget(popularity_top, content_top):
    return (pl.col("cutoff") <= popularity_top) | (
        (pl.col("cutoff") > 500) & (pl.col("cutoff") <= 500 + content_top)
    )


budget_results = []
for popularity_top in POPULARITY_BUDGETS:
    for content_top in CONTENT_BUDGETS:
        if popularity_top + content_top == 0:
            continue
        subset = scores.lazy().filter(in_budget(popularity_top, content_top))
        for ordering, expression in ORDERINGS.items():
            result = evaluate_retrieved_candidates(
                subset.with_columns(expression.alias("final_score")),
                feature_run=RUN,
                score_column="final_score",
                model_name=ordering,
            )
            budget_results.append(result.with_columns(
                pl.lit(ordering).alias("ordering"),
                pl.lit(popularity_top).alias("popularity_top"),
                pl.lit(content_top).alias("content_top"),
                pl.lit(popularity_top + content_top).alias("M"),
            ))
    print(f"popularity top {popularity_top}: done")

budget_results = pl.concat(budget_results).drop("model")
budget_results.write_csv(RESULTS / f"pipeline_budget_grid_{impression_tag}.csv")

## 5.1 Candidate Recall and Pipeline nDCG@10 by Budget

Rows are the popularity budget `P`, columns the content budget `C`. Recall can only rise
as the budget grows; ranking quality can fall if extra candidates are mostly noise the
reranker cannot separate.

In [ ]:
def budget_table(metric, ordering):
    return (
        budget_results.filter(pl.col("ordering") == ordering)
        .pivot(on="content_top", index="popularity_top", values=metric)
        .sort("popularity_top")
        .select("popularity_top", *[str(budget) for budget in CONTENT_BUDGETS])
    )


print("Candidate recall (share of known clicks retrieved):")
display(budget_table("candidate_recall", "generator_plus_reranker"))
print(f"Pipeline {SELECTION_METRIC} with the reranker:")
display(budget_table(SELECTION_METRIC, "generator_plus_reranker"))
print(f"Pipeline {SELECTION_METRIC} without the reranker:")
display(budget_table(SELECTION_METRIC, "generator_only"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, (metric, title) in zip(axes, [("candidate_recall", "Candidate recall"),
                                       (SELECTION_METRIC, f"{SELECTION_METRIC} with reranker")]):
    table = budget_table(metric, "generator_plus_reranker")
    values = table.drop("popularity_top").to_numpy()
    image = axis.imshow(values, cmap="viridis", aspect="auto", origin="lower")
    axis.set_xticks(range(len(table.columns) - 1), table.columns[1:])
    axis.set_yticks(range(table.height), table["popularity_top"].to_list())
    axis.set_xlabel("Content budget C")
    axis.set_ylabel("Popularity budget P")
    axis.set_title(title)
    for (row, column), value in np.ndenumerate(values):
        if value is not None and not np.isnan(value):
            axis.text(column, row, f"{value:.3f}", ha="center", va="center", fontsize=8, color="white")
    fig.colorbar(image, ax=axis, shrink=0.8)
plt.tight_layout()
plt.show()

## 5.2 Choosing the Budget

Two choices are reported:

- **Best**: the budget with the highest pipeline nDCG@10
- **Efficient**: the smallest `M` within `TOLERANCE` of the best. Every extra candidate
  costs feature computation and scoring at serving time, so this is often the better pick.

In [ ]:
reranked = budget_results.filter(pl.col("ordering") == "generator_plus_reranker")
best = reranked.sort([SELECTION_METRIC, "M"], descending=[True, False]).row(0, named=True)
efficient = (
    reranked.filter(pl.col(SELECTION_METRIC) >= best[SELECTION_METRIC] - TOLERANCE)
    .sort(["M", SELECTION_METRIC], descending=[False, True])
    .row(0, named=True)
)
for label, row in (("Best", best), ("Efficient", efficient)):
    print(f"{label}: P={row['popularity_top']}, C={row['content_top']} (M={row['M']}) -> "
          f"{SELECTION_METRIC}={row[SELECTION_METRIC]:.4f}, candidate recall={row['candidate_recall']:.4f}")

# Lift from adding the reranker at the chosen budget
chosen = budget_results.filter(
    (pl.col("popularity_top") == efficient["popularity_top"]) & (pl.col("content_top") == efficient["content_top"])
)
display(chosen.select("ordering", "M", "candidate_recall", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

## 5.3 Budget Curve

nDCG@10 against total budget `M` for each content share, with and without the reranker.

In [ ]:
fig, axis = plt.subplots(figsize=(8, 4.5))
for content_top in CONTENT_BUDGETS:
    for ordering, style in (("generator_plus_reranker", "-o"), ("generator_only", ":")):
        line = budget_results.filter((pl.col("content_top") == content_top) & (pl.col("ordering") == ordering)).sort("M")
        if line.height:
            label = f"C={content_top}" if ordering == "generator_plus_reranker" else None
            axis.plot(line["M"], line[SELECTION_METRIC], style, markersize=3, label=label, alpha=0.9 if label else 0.4)
axis.set_xlabel("Total candidates M = P + C")
axis.set_ylabel(SELECTION_METRIC)
axis.set_title("Solid: with reranker; dotted: generator order only")
axis.legend(title="Content budget", fontsize=8)
plt.tight_layout()
plt.show()

# 6. Interpretation

- **Candidate recall** is the ceiling: a click that neither retriever found can never be
  ranked. If recall is low at every budget, improving candidate generation matters more
  than the reranker.
- **The reranker's lift** is the gap between the two orderings at the same budget.
- **Pipeline vs logged-slate scores**: the reranking notebooks evaluate logged slates,
  where every candidate was shown and is labelled. Pipeline scores are lower because
  unretrieved clicks count as misses and unshown retrieved articles can never count as
  clicks. The two numbers answer different questions and shouldn't be compared directly.
- Choose the budget on validation. Re-scoring test candidates for the chosen budget is the
  final step.